# Lesson 3 — A mutually exclusive family is a dollar sold in pieces

When the exchange marks an event mutually exclusive, exactly one outcome resolves YES. Owning every outcome owns a guaranteed dollar, so what the pieces cost is a direct reading of whether the prices admit a probability.

**The rule.** `Σ p_i = 1 for a coherent family; Σ ask_i < 1 is a Dutch book before fees`

**When it holds.** Only when the exchange's own mutually_exclusive flag is set. That flag is the licence for the sum, not our arithmetic over the strikes.

**When it fails.** Buckets need not tile. Inferring exclusivity from floor and cap values asserts a claim the venue did not make, and a family with a gap in it has no reason to sum to anything.

| | |
|---|---|
| Lesson id | `basket` |
| Pane it appears on | `universe` (panes carry more than one lesson) |
| Code it is about | `modules/coherence/views.py` |
| Tests that go red if it stops being true | `tests/test_coherence_observe.py` |
| Pane shipped | yes |

Every cell below runs against the real kernel. Nothing here is a re-implementation:
a number this notebook prints is the number the engine would produce for the same
input. The recorded Kalshi payloads come from `tests/fixtures/coherence/`.

In [ ]:
import json
import sys
from decimal import Decimal
from pathlib import Path

# This notebook lives in notebooks/coherence_lab/ and imports the kernel two
# levels up. Found by walking upward rather than by counting parents, so the
# notebook runs from its own directory or from Part2_Infrastructure.
HERE = Path.cwd().resolve()
ROOT = next((path for path in (HERE, *HERE.parents) if (path / "modules" / "coherence" / "kernel").is_dir()), None)
if ROOT is None:
    raise SystemExit(f"no coherence kernel above {HERE}: open this notebook from inside Part2_Infrastructure")
sys.path.insert(0, str(ROOT))

FIXTURES = ROOT / "tests" / "fixtures" / "coherence"


def fixture(name: str) -> dict:
    """One recorded Kalshi response, envelope and all, exactly as it was sent.

    These are captures, not mocks. Where a number below looks odd it is because
    the exchange quoted it, and `tools/capture_kalshi_fixtures.py` re-records
    them.
    """
    return json.loads((FIXTURES / f"{name}.json").read_text(encoding="utf-8"))


print(f"kernel root       {ROOT}")
print(f"recorded fixtures {FIXTURES.is_dir()}")

## 1. A real mutually exclusive family

In [ ]:
from modules.coherence.drivers.kalshi_parse import parse_event
from modules.coherence.kernel import closedform, kelly
from modules.coherence.kernel.book import Book, Level
from modules.coherence.kernel.constraints import rows_for
from modules.coherence.kernel.costs import FeeSchedule
from modules.coherence.kernel.lattice import build_component

SCHEDULE = FeeSchedule()
event = parse_event(fixture("event_mee")["body"])
component = build_component(event)
books = {market.ticker: market.top for market in event.markets}

print(f"  {event.title} ({event.event_ticker})")
print(f"  the exchange marks it mutually exclusive: {event.mutually_exclusive}")
print(f"  settlement sources: {event.settlement_sources}")
print()
asks = Decimal(0)
for node in component.nodes:
    quote = books[node.ticker]
    asks += quote.best_yes_ask
    print(f"    {node.label:<22} bid {quote.best_yes_bid}  ask {quote.best_yes_ask}")
print()
print(f"  buying every outcome costs {asks} for the dollar exactly one of them pays")

## 2. The certificate on the quotes as recorded

In [ ]:
rows = rows_for(component, books)
certificate = closedform.solve(component, rows, SCHEDULE)
print(certificate.render_text())

## 3. One leg cheaper, and the dollar goes on sale

In [ ]:
# Move one leg's offer down until the basket costs under a dollar. Nothing else
# changes: the same flag, the same states, the same fee model.
CHEAPER = dict(books)
target = component.nodes[2]
CHEAPER[target.ticker] = Book(
    ticker=target.ticker,
    yes_bids=(Level(Decimal("0.3000"), 5_000),),
    no_bids=(Level(Decimal("0.4800"), 5_000),),
)
cheap_asks = sum((CHEAPER[node.ticker].best_yes_ask for node in component.nodes), Decimal(0))
print(f"  {target.label} now offered at {CHEAPER[target.ticker].best_yes_ask}; the basket costs {cheap_asks}")
print()
print(closedform.solve(component, rows_for(component, CHEAPER), SCHEDULE).render_text())

## 4. Sizing it: the arbitrage and the growth-optimal plan are not the same trade

In [ ]:
# Sizing the family is not the scalar formula repeated. Exactly one outcome pays,
# so a dollar on one is partly a hedge for the dollar on another.
mids = {node.ticker: books[node.ticker].mid for node in component.nodes}
mass = sum(mids.values(), Decimal(0))
plan = kelly.solve([
    kelly.Candidate(node.ticker, node.label, mids[node.ticker] / mass, CHEAPER[node.ticker].best_yes_ask)
    for node in component.nodes
])

print(f"  {plan.detail}")
print()
for stake in plan.stakes:
    marker = "stake" if stake.admitted else "  -  "
    quantised = stake.probability.quantize(Decimal("0.0001"))
    print(f"  {marker} {stake.label:<22} q {quantised} @ {stake.price}  quarter-Kelly {stake.fraction.quantize(Decimal('0.0001'))}")
print()
print(f"  basket cost         {plan.basket_cost}")
print(f"  riskless log growth {plan.riskless_growth.quantize(Decimal('0.000001'))}   (equal contracts of every outcome, certain)")
print(f"  full Kelly growth   {plan.full_growth_rate.quantize(Decimal('0.000001'))}   (stakes in proportion to the measure)")
print(f"  quarter Kelly       {plan.growth_rate.quantize(Decimal('0.000001'))}   (the shipped default, shrinkage {plan.shrinkage})")
print(f"  worst-case wealth   {plan.worst_case_wealth.quantize(Decimal('0.0001'))} of the bankroll, so this plan can lose")
print()
print("  Full Kelly grows faster than the arbitrage precisely because it is taking a risk")
print("  the arbitrage refuses. The quarter does not, and that is the trade being made:")
print("  Kelly's growth curve is flat near the optimum and steep past it, so over-betting")
print("  costs far more than under-betting, and q here is read off a moving book.")
print()
print("  The certificate answers 'what can I be paid for holding nothing?'. Kelly answers")
print("  'what maximises growth if my measure is right?'. They are different portfolios.")

## 5. The flag is the licence for the sum

In [ ]:
from dataclasses import replace

unflagged = build_component(replace(event, mutually_exclusive=False))
print(f"  the same five markets with the flag off: {len(rows_for(unflagged, books))} rows, against {len(rows)} with it")
for note in unflagged.notes:
    print(f"    {note}")
print()
print("  Buckets need not tile. Inferring exclusivity from floor and cap values asserts a")
print("  claim the venue did not make, and a family with a gap in it has no reason to sum")
print("  to anything.")